### Goal  

Build a Word2Vec‑based baseline for sentiment classification.

Train a Word2Vec model on tokenized reviews, convert each review into an averaged embedding, and evaluate simple classifiers.

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../src"))

import my_utils  # noqa: F401

In [2]:
labeled_df = pd.read_csv('../data/labeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)
unlabeled_df = pd.read_csv('../data/unlabeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)


print(f"labeled_df shape: {labeled_df.shape}")
print(f"unlabeled_df shape: {unlabeled_df.shape}")

labeled_df shape: (25000, 3)
unlabeled_df shape: (50000, 2)


### Using unlabeled data  

Word2Vec is an unsupervised model, so unlabeled reviews can be incorporated to improve embedding quality.

Training on both labeled and unlabeled data increases vocabulary coverage and produces more stable semantic representations.

In [3]:
all_reviews = pd.concat([labeled_df.review, unlabeled_df.review])
sentences = all_reviews.nlp.review_to_wordlist().tolist()

In [4]:
sentences[:2]

[['stuff',
  'going',
  'moment',
  'mj',
  'started',
  'listening',
  'music',
  'watching',
  'odd',
  'documentary',
  'watched',
  'wiz',
  'watched',
  'moonwalker',
  'maybe',
  'want',
  'get',
  'certain',
  'insight',
  'guy',
  'thought',
  'really',
  'cool',
  'eighties',
  'maybe',
  'make',
  'mind',
  'whether',
  'guilty',
  'innocent',
  'moonwalker',
  'part',
  'biography',
  'part',
  'feature',
  'film',
  'remember',
  'going',
  'see',
  'cinema',
  'originally',
  'released',
  'subtle',
  'messages',
  'mj',
  'feeling',
  'towards',
  'press',
  'also',
  'obvious',
  'message',
  'drugs',
  'bad',
  'kay',
  'visually',
  'impressive',
  'course',
  'michael',
  'jackson',
  'unless',
  'remotely',
  'like',
  'mj',
  'anyway',
  'going',
  'hate',
  'find',
  'boring',
  'may',
  'call',
  'mj',
  'egotist',
  'consenting',
  'making',
  'movie',
  'mj',
  'fans',
  'would',
  'say',
  'made',
  'fans',
  'true',
  'really',
  'nice',
  'actual',
  'feature

In [5]:
from gensim.models import Word2Vec

vector_size = 200
window = 5

w2v_model = Word2Vec(
    sentences,
    vector_size=vector_size,
    window=window,
    min_count=5,
    workers=4,
    sg=1,              # skip-gram 
    seed=42
)


In [6]:
w2v_model.save("../data/word2vec.model")

In [7]:
for word in sentences[0]:
    if word in w2v_model.wv:
        # print the word and first 5 dimensions of its vector representation
        print(f"{word}: {w2v_model.wv[word][:5]} ...")
    else:
        print(f"{word}: not in vocabulary")

stuff: [0.26265487 0.43405965 0.2212945  0.00938103 0.04736961] ...
going: [-0.1092691  -0.03357348  0.07824143  0.07793499 -0.43408975] ...
moment: [-0.04702201 -0.00048006  0.11674121  0.23326403 -0.134997  ] ...
mj: [0.0743283  0.07650959 0.1154236  0.05578576 0.10649937] ...
started: [ 0.02089911  0.06354674  0.32632253  0.2970926  -0.01103074] ...
listening: [-0.08361873  0.0347803   0.3456848   0.15415213  0.09717152] ...
music: [0.3404138  0.30344027 0.08128744 0.2577308  0.03814802] ...
watching: [ 0.06029228  0.06430819  0.24397841  0.13304397 -0.06277066] ...
odd: [ 0.47103927  0.3258161   0.2739141  -0.20409425  0.11569505] ...
documentary: [ 0.19500086  0.5323268   0.16095254 -0.14369148  0.33554834] ...
watched: [ 0.07289262 -0.10939842  0.15880005  0.08113026 -0.1806803 ] ...
wiz: [ 0.10675739  0.14318557  0.07837459  0.16529728 -0.03180369] ...
watched: [ 0.07289262 -0.10939842  0.15880005  0.08113026 -0.1806803 ] ...
moonwalker: [0.19889694 0.27710944 0.14135644 0.14057

In [8]:
labeled_wordlist = labeled_df.review.nlp.review_to_wordlist()
labeled_sentences = labeled_wordlist.tolist()

len(labeled_sentences)

25000

To convert each review into a single vector, I compute the mean embedding of all words in the review that appear in the Word2Vec vocabulary.

This approach is simple but surprisingly effective:

It smooths out noise

It captures the overall semantic “direction” of the review

It produces a fixed‑size vector for any review length

The resulting matrix has shape: **(n_reviews, embedding_dim)**

In [9]:
import numpy as np

def review_to_vec(words, model, dim=vector_size):
    valid_words = [w for w in words if w in model.wv]

    if not valid_words:
        return np.zeros(dim)
    
    return np.mean(model.wv[valid_words], axis=0)

X_w2v = np.vstack([review_to_vec(words, w2v_model, vector_size) for words in labeled_sentences])


In [10]:
X_w2v.shape

(25000, 200)

In [11]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X_w2v, labeled_df.sentiment, test_size=0.2, random_state=42
)

### Baseline Models

I evaluate three standard classifiers on the averaged Word2Vec embeddings:

- Logistic Regression

- Linear SVM (LinearSVC)

- Random Forest

These models provide a clear comparison to the TF‑IDF baseline and help quantify the value of dense semantic embeddings.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

evaluators = [
    (LogisticRegression(max_iter=500), 'LogisticRegression'),
    (LinearSVC(), 'LinearSVC'),
    (RandomForestClassifier(random_state=42, max_depth=10), 'RandomForestClassifier')
]

In [13]:
for tup in evaluators:
    (clf, eval_class) = tup
    
    print(f'Evaluating {eval_class}')
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)    
    print(f'{eval_class} sentiment score: {score}')
    print("\n")

Evaluating LogisticRegression
LogisticRegression sentiment score: 0.8852


Evaluating LinearSVC
LinearSVC sentiment score: 0.885


Evaluating RandomForestClassifier
RandomForestClassifier sentiment score: 0.848




### Summary

Word2Vec provides a meaningful alternative representation of text that captures semantic relationships beyond simple word frequency.
In this baseline, averaged Word2Vec embeddings achieve competitive performance and form a strong foundation for further improvements, such as:

- training on labeled + unlabeled reviews

- increasing embedding dimensionality

This completes the second major baseline for the project.